In [ ]:
import pandas as pd
import numpy as np

from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_log_error as RMSE

data = pd.read_csv("train.csv")

target_thing = np.log1p(data["hours_spent"])

input_data = data.drop(columns=[
    "project_id",
    "hours_spent",
    "log_hours",
    "hours_bucket"
])

qualitative_features = input_data.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

for col in qualitative_features:
    input_data[col] = input_data[col].fillna("Unknown").astype(str)

input_training, input_validation, output_training, output_validation = train_test_split(
    input_data,
    target_thing,
    test_size=0.2,
    random_state=42
)

BASE_PARAMS = {
    "iterations": 1000,
    "learning_rate": 0.01,
    "depth": 6,
    "loss_function": "RMSE",
    "verbose": False,
    "early_stopping_rounds": 100
}

SEEDS = [1, 23, 42, 100, 999]

def evaluate_model(params, seed):

    model_params = params.copy()
    model_params["random_seed"] = seed

    model = CatBoostRegressor(**model_params)

    model.fit(
        input_training,
        output_training,
        cat_features=qualitative_features,
        eval_set=(input_validation, output_validation),
        verbose=False
    )

    predictions = np.expm1(model.predict(input_validation))
    predictions = np.maximum(predictions, 0)

    true_hours = np.expm1(output_validation)

    score = RMSE(true_hours, predictions)

    return score, model

def test_config(name, params):

    scores = []
    iterations = []

    for seed in SEEDS:

        score, model = evaluate_model(params, seed)

        scores.append(score)
        iterations.append(model.get_best_iteration())

        print(
            f"{name:25s} | "
            f"Seed {seed:3d} | "
            f"RMSE {score:.6f}"
        )

    return {
        "configuration": name,
        "mean": np.mean(scores),
        "std": np.std(scores, ddof=1),
        "best": np.min(scores),
        "worst": np.max(scores),
        "mean_iteration": np.mean(iterations)
    }


def show_results(results, experiment_name):

    df = pd.DataFrame(results)

    df = df.sort_values("mean").reset_index(drop=True)

    print(experiment_name)

    print(df.to_string(index=False))

    print("\nBest configuration:")
    print(df.iloc[0].to_string())

    return df

In [ ]:
results = []

configs = [
    ("SymmetricTree", {
        "grow_policy": "SymmetricTree"
    }),

    ("Depthwise", {
        "grow_policy": "Depthwise"
    }),

    ("Lossguide-8", {
        "grow_policy": "Lossguide",
        "max_leaves": 8
    }),

    ("Lossguide-16", {
        "grow_policy": "Lossguide",
        "max_leaves": 16
    }),

    ("Lossguide-32", {
        "grow_policy": "Lossguide",
        "max_leaves": 32
    }),

    ("Lossguide-64", {
        "grow_policy": "Lossguide",
        "max_leaves": 64
    })
]

for name, config in configs:

    params = BASE_PARAMS.copy()
    params.update(config)

    result = test_config(name, params)
    results.append(result)

grow_results = show_results(results, "GROW POLICY RESULTS")

grow_results.to_csv("grow_policy_results.csv", index=False)

In [ ]:
results = []

configs = [
    ("Bayesian-0", {
        "bootstrap_type": "Bayesian",
        "bagging_temperature": 0
    }),

    ("Bayesian-0.5", {
        "bootstrap_type": "Bayesian",
        "bagging_temperature": 0.5
    }),

    ("Bayesian-1", {
        "bootstrap_type": "Bayesian",
        "bagging_temperature": 1
    }),

    ("Bayesian-2", {
        "bootstrap_type": "Bayesian",
        "bagging_temperature": 2
    }),

    ("Bayesian-5", {
        "bootstrap_type": "Bayesian",
        "bagging_temperature": 5
    }),

    ("Bernoulli-0.5", {
        "bootstrap_type": "Bernoulli",
        "subsample": 0.5
    }),

    ("Bernoulli-0.7", {
        "bootstrap_type": "Bernoulli",
        "subsample": 0.7
    }),

    ("Bernoulli-0.8", {
        "bootstrap_type": "Bernoulli",
        "subsample": 0.8
    }),

    ("Bernoulli-0.9", {
        "bootstrap_type": "Bernoulli",
        "subsample": 0.9
    }),

    ("MVS-0.5", {
        "bootstrap_type": "MVS",
        "subsample": 0.5
    }),

    ("MVS-0.7", {
        "bootstrap_type": "MVS",
        "subsample": 0.7
    }),

    ("MVS-0.8", {
        "bootstrap_type": "MVS",
        "subsample": 0.8
    }),

    ("MVS-0.9", {
        "bootstrap_type": "MVS",
        "subsample": 0.9
    }),

    ("No-bootstrap", {
        "bootstrap_type": "No"
    })
]

for name, config in configs:

    params = BASE_PARAMS.copy()
    params.update(config)

    result = test_config(name, params)
    results.append(result)

bootstrap_results = show_results(results, "BOOTSTRAP RESULTS")

bootstrap_results.to_csv("bootstrap_results.csv", index=False)

In [ ]:
results = []

for strength in [0, 0.1, 0.25, 0.5, 1, 2, 5, 10]:

    params = BASE_PARAMS.copy()
    params["random_strength"] = strength

    result = test_config(
        f"Random-{strength}",
        params
    )

    results.append(result)

random_results = show_results(results, "RANDOM STRENGTH RESULTS")

random_results.to_csv("random_strength_results.csv", index=False)

In [ ]:
results = []

for boosting in ["Plain", "Ordered"]:

    params = BASE_PARAMS.copy()
    params["boosting_type"] = boosting

    result = test_config(
        boosting,
        params
    )

    results.append(result)

boosting_results = show_results(results, "BOOSTING TYPE RESULTS")

boosting_results.to_csv("boosting_results.csv", index=False)

In [ ]:
results = []

for method in ["Newton", "Gradient"]:

    for iterations in [1, 3, 5, 10]:

        params = BASE_PARAMS.copy()

        params["leaf_estimation_method"] = method
        params["leaf_estimation_iterations"] = iterations

        name = f"{method}-{iterations}"

        result = test_config(name, params)

        results.append(result)

leaf_results = show_results(results, "LEAF ESTIMATION RESULTS")

leaf_results.to_csv("leaf_estimation_results.csv", index=False)

In [ ]:
results = []

for borders in [32, 64, 128, 254]:

    params = BASE_PARAMS.copy()
    params["border_count"] = borders

    result = test_config(
        f"Border-{borders}",
        params
    )

    results.append(result)

border_results = show_results(results, "BORDER COUNT RESULTS")

border_results.to_csv("border_count_results.csv", index=False)

In [ ]:
all_results = pd.concat([
    grow_results,
    bootstrap_results,
    random_results,
    boosting_results,
    leaf_results,
    border_results
], ignore_index=True)

all_results = all_results.sort_values("mean")

print("OVERALL EXPERIMENT RESULTS")

print(all_results.to_string(index=False))

all_results.to_csv("all_experiment_results.csv", index=False)